In [ ]:
from faster_whisper import WhisperModel

In [ ]:
model_size = "large-v3"
# model_size = "medium"

#? Run on GPU with FP16
# model = WhisperModel(model_size, device="cuda", compute_type="float16", num_workers=10, cpu_threads=16 )

#? or run on GPU with INT8
# model = WhisperModel(model_size, device="cuda", compute_type="int8_float16")

#? or run on CPU with INT8
# model = WhisperModel(model_size, device="cpu", compute_type="int8")
model = WhisperModel(model_size, device="cpu", compute_type="int8", num_workers=10, cpu_threads=16 )


In [ ]:
# file_path = "../data/audio_samples/sample.mp3"
file_path = "../data/audio_samples/sample2.wav"
# file_path = "../data/audio_samples/long_sample.mp3"

from pydub import AudioSegment
import numpy as np

audio_segment = AudioSegment.from_file(file_path, format=file_path.split(".")[-1])
# audio_segment = np.array(audio_segment.get_array_of_samples())
# convert to expected format
print(audio_segment.frame_rate)
print(audio_segment.sample_width)
print(audio_segment.channels)

if audio_segment.frame_rate != 16000: # 16 kHz
    audio_segment = audio_segment.set_frame_rate(16000)
if audio_segment.sample_width != 2:   # int16
    audio_segment = audio_segment.set_sample_width(2)
if audio_segment.channels != 1:       # mono
    audio_segment = audio_segment.set_channels(1)        
arr = np.array(audio_segment.get_array_of_samples())
arr = arr.astype(np.float32)/32768.0

In [ ]:
segments, info = model.transcribe(arr, beam_size=1)
a = "".join([x.text for x in segments])
a

In [ ]:
print("Detected language '%s' with probability %f" % (info.language, info.language_probability))

for segment in segments:
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))